### **Цель:** собрать модели ниже, и проверить их эффективность в ансамбле.

![diagram](./attach/issc-diagram-1.svg)

In [9]:
from pathlib import Path
import pandas as pd
import collections
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
import torch
from random import seed
from sklearn.model_selection import train_test_split, KFold

from importer import *

In [10]:
PATH_ACTIVITY = Path('../DATA/activity/')
PATH_METAINFO = Path('../DATA/metainfo/name+type+ghedonic+gender.csv')
PATH_SPECTRAL = Path('../DATA/spectral/')

### 1. Конвертируем данные EEG-Cutter в нужный формат

In [11]:
activity_df_columns = []

for i in range(2): # На каждое предъявление у нас 3 файла,
                   # 3-й мы будем вычитать из 1-го и из 2-го
    for zn in ZONE_NAMES:
        activity_df_columns.append(f'{zn}_{i+1}')

activity_df = pd.DataFrame(columns=activity_df_columns)

subjects = Path.iterdir(PATH_ACTIVITY)
iii = 0

for subject in sorted(subjects):
    if subject.is_dir():
        files = sorted(filter(lambda f: str(f).endswith('.csv'), Path.iterdir(subject)))
        print(subject, len(files))
        
        for i in range(0, len(files), 3):
            p0 = convert_eegc_activity(files[i])
            p10 = convert_eegc_activity(files[i+1])
            p100 = convert_eegc_activity(files[i+2])

            delta1 = {k: p0[k] - p100[k] for k in p0}
            delta2 = {k: p10[k] - p100[k] for k in p10}

            for k, v in delta1.items():
                activity_df.at[iii, f'{k}_1'] = v
            for k, v in delta2.items():
                activity_df.at[iii, f'{k}_2'] = v

            iii += 1

activity_df

../DATA/activity/Bmet_001 72
../DATA/activity/Bmet_007 18
../DATA/activity/Bmet_011 21
../DATA/activity/Met_003 36
../DATA/activity/Met_004 36
../DATA/activity/Met_005 36
../DATA/activity/Met_006 36
../DATA/activity/Met_009 18
../DATA/activity/Met_010 30
../DATA/activity/Met_011 21
../DATA/activity/Met_013 18
../DATA/activity/NOTUSEDBYSHI 0


,LF_1,MF_1,RF_1,LT_1,MC_1,RT_1,LP_1,MP_1,RP_1,LF_2,MF_2,RF_2,LT_2,MC_2,RT_2,LP_2,MP_2,RP_2
0,0.407766,0.004426,-0.003005,0.082742,-0.07042,-0.024272,0.11594,0.01524,0.083118,-0.249729,0.010985,-0.152518,-0.096465,-0.050107,-0.092899,0.10161,0.143787,-0.057852
1,0.127947,0.108755,0.017622,0.010425,0.043905,0.170563,-0.068743,-0.199585,-0.058262,0.23326,-0.124707,0.045846,0.021066,0.041013,-0.086686,-0.10246,0.014511,0.082752
2,0.035437,-0.068243,0.121052,0.02179,0.054783,-0.120041,0.044528,-0.273911,0.125953,-0.106089,-0.336216,-0.071849,-0.064039,0.169695,0.049142,-0.024852,-0.065088,-0.02176
3,0.420544,-0.395108,0.124535,0.073632,-0.030876,0.045107,-0.072702,-0.419326,0.039387,0.521661,-0.256036,0.395502,0.030065,-0.058379,0.066512,0.044916,-0.165503,-0.012589
4,0.426035,-0.090777,0.196875,0.113779,-0.132729,0.049885,0.011725,-0.156083,-0.007535,-0.388224,-0.058263,-0.124829,-0.165576,0.097775,-0.071409,-0.059249,-0.038213,-0.064399
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,0.088191,0.010942,0.022813,0.023624,0.032087,0.15901,0.145301,0.112881,0.317546,-0.104223,-0.009773,0.366493,0.131785,0.043586,-0.309942,0.271386,-0.025747,-0.089629
110,-0.480998,-0.308856,0.290684,0.332032,-0.028832,-0.230101,-0.058325,-0.033538,-0.10915,0.092222,-0.226209,0.168962,0.382058,-0.041152,0.652146,-0.138042,0.120163,-0.142922
111,-0.123498,-0.056899,0.172386,0.194827,0.053798,-0.449821,0.05717,-0.066195,0.025463,-0.073922,0.015785,0.103151,0.270246,-0.026154,-0.038716,0.10544,0.099473,0.460406
112,-0.220663,0.121912,-0.261202,0.107631,0.18335,-0.241005,0.609047,0.042978,-0.902252,-0.578094,0.006552,0.192032,0.483955,-0.031218,0.101183,0.237179,-0.043493,-0.020428


### 2. Подготавливаем метаданные и спектральные характеристики

Метаданные (в т.ч. категории запахов), в сочетании с активностями областей по sLORETA, используются в первом датасете, а спектральные характреистики (опять таки в сочетании с метаданными) нужны для второго.

In [12]:
metainfo = pd.read_csv(PATH_METAINFO)

metainfo

,subject,trial,scent_name,scent_type,ghedonic,gender
0,Bmet_001,1,Ладан,Древесные,1,1
1,Bmet_001,2,Розмарин,Хвойные,-1,1
2,Bmet_001,3,Чайное дерево,Хвойные,1,1
3,Bmet_001,4,Нероли,Цветочные,-1,1
4,Bmet_001,5,Майоран,Травянистые / Гурманские,-2,1
...,...,...,...,...,...,...
109,Met_013,2,Сладкий апельсин,Цитрусовые,1,1
110,Met_013,3,Герань,Цветочные,0,1
111,Met_013,4,Бергамот,Цитрусовые,2,1
112,Met_013,5,Лаванда,Цветочные,1,1


In [13]:
counts = collections.Counter()

for t, c in metainfo['scent_type'].value_counts().items():
    ts = str(t).split('/')
    
    for s in ts:
        ss = s.strip()
        counts[ss] += c

type_c = pd.Series(counts)

type_c.sort_values()

Землистые       2
Мятные          7
Гурманские     15
Травянистые    15
Древесные      17
Хвойные        20
Цветочные      23
Цитрусовые     29
dtype: int64

In [14]:
metainfo['scent_type_split'] = metainfo['scent_type'].apply(
    lambda ts: [t.strip() for t in ts.split('/')]
)

mlb = MultiLabelBinarizer()
groups = mlb.fit_transform(metainfo['scent_type_split'])
groups_df = pd.DataFrame(groups, columns=mlb.classes_)

groups_df

,Гурманские,Древесные,Землистые,Мятные,Травянистые,Хвойные,Цветочные,Цитрусовые
0,0,1,0,0,0,0,0,0
1,0,0,0,0,0,1,0,0
2,0,0,0,0,0,1,0,0
3,0,0,0,0,0,0,1,0
4,1,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...
109,0,0,0,0,0,0,0,1
110,0,0,0,0,0,0,1,0
111,0,0,0,0,0,0,0,1
112,0,0,0,0,0,0,1,0


In [15]:
spectral_df_columns = []

for rhythm in ('δ', 'θ', 'α', 'β', 'γ'):
    # Да, именно в таком порядке!
    # См. https://ru.wikipedia.org/wiki/Ритмы_головного_мозга
    for zn in ZONE_NAMES:
        spectral_df_columns.append(f'{rhythm}_{zn}')

spectral_df = pd.DataFrame(columns=spectral_df_columns).astype('float32')

subjects = Path.iterdir(PATH_SPECTRAL)

for i, r in enumerate(('delta', 'theta', 'alpha', 'beta', 'gamma')):
    spectre = pd.read_csv(PATH_SPECTRAL / (r + '.csv'))

    spectral_df.iloc[:, i*9 : i*9+9] = spectre.iloc[:, 2:11].values

spectral_df

/tmp/ipykernel_523199/4124933630.py:16: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[1.05267017e+03 1.02860034e+03 6.19647461e+02 1.66636145e+03
 1.11820471e+03 5.34555359e+02 8.12467163e+02 7.66041687e+02
 1.63260620e+03 1.00733319e+03 2.06729443e+03 8.19500854e+02
 2.38282090e+04 1.66173344e+05 1.93805188e+05 1.09007086e+05
 3.09718125e+05 2.28121531e+05 1.97585516e+05 3.01948875e+05
 6.60149438e+05 1.03637556e+06 1.00429883e+05 2.37805938e+04
 7.68161774e+01 8.00840530e+01 5.30507355e+01 7.72419052e+01
 5.39906349e+01 6.45988312e+01 3.42844696e+01 2.61776524e+01
 2.78865910e+01 2.71302700e+01 3.28275757e+01 2.98802700e+01
 3.37602043e+01 1.09623756e+02 5.55230751e+01 1.26869408e+02
 8.25429077e+01 7.11396256e+01 6.25417328e+01 9.22183533e+01
 7.25955582e+01 1.18197975e+02 9.15843658e+01 5.77279739e+01
 4.21325500e+02 2.81964050e+02 1.37455612e+02 1.90302155e+02
 1.30559708e+02 9.46272888e+01 2.66327229e+01 2.8

,δ_LF,δ_MF,δ_RF,δ_LT,δ_MC,δ_RT,δ_LP,δ_MP,δ_RP,θ_LF,...,β_RP,γ_LF,γ_MF,γ_RF,γ_LT,γ_MC,γ_RT,γ_LP,γ_MP,γ_RP
0,394.282043,1052.670166,340.846619,417.891632,624.345215,356.835876,847.289734,544.589966,817.558960,59.074821,...,37.029888,199.981049,71.096779,32.656998,64.772011,1200.609009,52.518333,961.223938,958.408447,1641.589722
1,762.995667,1028.600342,323.704681,523.671509,3088.326904,236.548523,3412.561523,2571.625977,4525.356934,80.203697,...,93.544815,220.646896,78.612442,32.818657,75.055435,1240.861206,62.599491,1048.204224,1000.914795,1702.527710
2,624.088623,619.647461,214.890778,362.819153,2063.662842,282.017670,2571.751465,2155.573486,2894.437012,51.422115,...,78.263687,277.299713,98.526665,36.027576,97.729782,1324.911621,68.919403,1248.845947,1095.593994,1826.238159
3,1850.210327,1666.361450,584.639832,965.549133,5260.854980,560.366577,7790.125000,5372.615723,8168.583008,149.589188,...,160.597122,501.910645,176.329025,54.761299,202.971405,2008.698364,63.239769,2481.108643,1757.602661,2808.519043
4,1587.042114,1118.204712,321.446777,684.041443,4352.611328,409.999542,4622.062988,3480.548584,6394.530762,151.863632,...,170.729218,522.595520,182.873444,52.384823,259.869598,1875.250000,72.528259,2702.287109,1709.924683,2638.802734
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,52.529495,32.003029,57.564640,79.792259,26.361336,282.841888,27.592903,21.467812,75.760773,21.167177,...,20.194298,19.207043,9.158166,14.601722,30.616104,4.883770,12.496562,6.511979,5.536946,14.388382
110,52.660519,37.597427,64.193718,77.507248,32.133537,119.770058,36.273335,21.740889,58.752228,25.902210,...,20.669523,15.640918,6.016054,11.199177,16.251923,4.368326,16.239906,5.572407,4.487237,13.464450
111,48.808262,34.186184,48.719093,52.038326,22.092644,51.083508,24.270288,17.192556,48.512646,21.284058,...,21.487478,23.787455,9.715988,13.419046,23.677118,5.281605,14.337770,7.586077,5.050881,15.914467
112,53.893646,32.969524,88.254791,61.386993,28.986883,49.418941,35.856918,28.828800,56.903214,20.861874,...,29.863012,28.553324,14.476421,20.893141,66.449112,7.059452,19.008575,9.604828,6.554445,22.771816


Последняя проверка, что ничего не поехало по одному месту:

In [16]:
print(
    metainfo.shape[1],    # = 2 + 2 + 1 + 1 + 1
    groups_df.shape[1],   # = От 8 до 10
    activity_df.shape[1], # = CSV_PER_CAT * 9
    spectral_df.shape[1]  # = 5 * 9
)

assert(metainfo.shape[0] == groups_df.shape[0] == activity_df.shape[0] == spectral_df.shape[0])

7 8 18 45


### 3. Агрегируем датасеты для обоих версий модели

**ДАТАСЕТ 1:** пол + encode категории(-й) + средняя активность областей на sLORETA --> гедоническая оценка

**ДАТАСЕТ 2:** пол + encode категории(-й) + спектральные характеристики областей --> гедоническая оценка

In [17]:
dataset_input1 = pd.concat([
    metainfo['gender'],
    groups_df,
    activity_df
], axis=1).astype('float32')

dataset_input2 = pd.concat([
    metainfo['gender'],
    groups_df,
    spectral_df
], axis=1).astype('float32')

dataset_output = pd.concat([
    # metainfo['gender'],
    metainfo['ghedonic'].apply(lambda g: (g + 3.0) / 5.0),
]).astype('float32')

In [18]:
RANDOM_SEED = 80085 # Spoopy!!

torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
seed(RANDOM_SEED)

# print(dataset_input1.iloc[21])

In [19]:
train_input1, test_input1, train_output, test_output = train_test_split(
    dataset_input1, dataset_output, test_size=0.3, random_state=RANDOM_SEED
)

train_input2, test_input2, _, _ = train_test_split(
    dataset_input2, dataset_output, test_size=0.3, random_state=RANDOM_SEED
)

scaler1 = StandardScaler()
train_input1 = scaler1.fit_transform(train_input1)
test_input1 = scaler1.transform(test_input1)
scaler2 = StandardScaler()
train_input2 = scaler2.fit_transform(train_input2)
test_input2 = scaler2.transform(test_input2)

In [20]:
train_input1

array([[-1.661898  , -0.3806935 , -0.4438127 , ...,  0.27947778,
        -0.02593173,  0.15660825],
       [ 0.6017217 , -0.3806935 , -0.4438127 , ...,  0.27797696,
        -0.16836888, -0.8145145 ],
       [-1.661898  , -0.3806935 ,  2.253203  , ...,  0.6923419 ,
         0.66200376,  1.1877035 ],
       ...,
       [-1.661898  , -0.3806935 , -0.4438127 , ..., -1.453573  ,
        -0.240304  , -1.0091723 ],
       [ 0.6017217 , -0.3806935 , -0.4438127 , ...,  1.204727  ,
        -0.06404526, -0.1363709 ],
       [ 0.6017217 ,  2.626785  , -0.4438127 , ...,  0.49357003,
         0.8237692 ,  0.5413129 ]], shape=(79, 27), dtype=float32)

In [21]:
train_input2

array([[-1.661898  , -0.3806935 , -0.4438127 , ..., -0.33282265,
        -0.34024307, -0.39495024],
       [ 0.6017217 , -0.3806935 , -0.4438127 , ..., -0.33281213,
        -0.34007192, -0.391003  ],
       [-1.661898  , -0.3806935 ,  2.253203  , ..., -0.332817  ,
        -0.3401041 , -0.3898273 ],
       ...,
       [-1.661898  , -0.3806935 , -0.4438127 , ..., -0.33282393,
        -0.34025002, -0.39528495],
       [ 0.6017217 , -0.3806935 , -0.4438127 , ..., -0.33282718,
        -0.34027717, -0.39545208],
       [ 0.6017217 ,  2.626785  , -0.4438127 , ..., -0.3328154 ,
        -0.34013903, -0.3919569 ]], shape=(79, 54), dtype=float32)

In [22]:
train_output.astype(np.float32)

79     0.6
95     0.6
43     0.8
49     1.0
5      0.2
      ... 
10     0.6
106    0.8
39     0.8
109    0.8
97     1.0
Name: ghedonic, Length: 79, dtype: float32

### 4. Собственно, проверка датасетов на моделях!

Архитектура примерно такая же, как и в прошылй раз, покрмр на текущий момент

In [23]:
def learning_pipeline(
        model,
        train_input,
        train_output,
        test_input,
        test_output,
        comments=True
    ):
    train_input_t = torch.tensor(train_input, dtype=torch.float32)
    train_output_t = torch.tensor(train_output.values, dtype=torch.float32).unsqueeze(1)
    test_input_t = torch.tensor(test_input, dtype=torch.float32)
    test_output_t = torch.tensor(test_output.values, dtype=torch.float32).unsqueeze(1)

    criterion = torch.nn.MSELoss() 
    optimizer = torch.optim.Adam(model.parameters(), lr=0.00002, weight_decay=0.001)


    best_pred = None
    best_loss = 99.99
    best_e = -1

    for e in range(999999999):
        model.train()
        optimizer.zero_grad()
        
        train_pred = model(train_input_t)
        train_loss = criterion(train_pred, train_output_t)
        
        train_loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            test_pred = model(test_input_t)
            test_loss = criterion(test_pred, test_output_t)

            if test_loss.item() < best_loss:
                best_e = e
                best_loss = test_loss.item()
                best_pred = test_pred.clone()

        if e % 200 == 0:
            if comments:
                print(f'Потери на train на эпохе {e} = {train_loss.item()}')
                print(f'Потери на test на эпохе {e} = {test_loss.item()}\n')

            if e - best_e >= 2000:
                if comments:
                    print('Модель перестала учиться, this is the end...')

                print(f'Лучшая эпоха: {best_e} (потери на test: {best_loss})')
                break
    
    return (
        test_output_t.squeeze().numpy(),
        best_pred.squeeze().numpy(),
        best_loss
    )

In [24]:
DROPOUT_P = 0.1


class Dataset1Model(torch.nn.Module):
    def __init__(self):
        super(Dataset1Model, self).__init__()

        self.fc1 = torch.nn.Linear(27, 12)     # = Гендер (1) + Категории (8) + Активности областей (9*2)
        self.do1 = torch.nn.Dropout(DROPOUT_P) # Контрим переобучение на малом кол-ве данных
        self.fc2 = torch.nn.Linear(12, 6)      # Расширяем
        self.do2 = torch.nn.Dropout(DROPOUT_P) # Контрим переобучение на малом кол-ве данных
        self.out = torch.nn.Linear(6, 1)       # = Гедоническая оценка (1)

    def forward(self, x):
        x = torch.nn.functional.silu(self.fc1(x))
        x = self.do1(x)
        x = torch.nn.functional.silu(self.fc2(x))
        x = self.do2(x)
        x = self.out(x)

        return x

class Dataset2Model(torch.nn.Module):
    def __init__(self):
        super(Dataset2Model, self).__init__()

        self.fc1 = torch.nn.Linear(54, 16)     # = Гендер (1) + Категории (8) + Спектральные хар-ки (9*5)
        self.do1 = torch.nn.Dropout(DROPOUT_P) # Контрим переобучение на малом кол-ве данных
        self.fc2 = torch.nn.Linear(16, 8)      # Расширяем
        self.do2 = torch.nn.Dropout(DROPOUT_P) # Контрим переобучение на малом кол-ве данных
        self.out = torch.nn.Linear(8, 1)       # = Гедоническая оценка (1)

    def forward(self, x):
        x = torch.nn.functional.silu(self.fc1(x))
        x = self.do1(x)
        x = torch.nn.functional.silu(self.fc2(x))
        x = self.do2(x)
        x = self.out(x)

        return x

In [27]:
learning_pipeline(
    Dataset1Model(),
    train_input1,
    train_output,
    test_input1,
    test_output
)

Потери на train на эпохе 0 = 1.1738606691360474
Потери на test на эпохе 0 = 1.0682622194290161



Потери на train на эпохе 200 = 1.1477755308151245
Потери на test на эпохе 200 = 1.0438202619552612

Потери на train на эпохе 400 = 1.1204524040222168
Потери на test на эпохе 400 = 1.0197935104370117

Потери на train на эпохе 600 = 1.098256230354309
Потери на test на эпохе 600 = 0.9959567189216614

Потери на train на эпохе 800 = 1.076558232307434
Потери на test на эпохе 800 = 0.9723265767097473

Потери на train на эпохе 1000 = 1.042858362197876
Потери на test на эпохе 1000 = 0.9489963054656982

Потери на train на эпохе 1200 = 1.0140366554260254
Потери на test на эпохе 1200 = 0.9259944558143616

Потери на train на эпохе 1400 = 0.9806649088859558
Потери на test на эпохе 1400 = 0.903328001499176

Потери на train на эпохе 1600 = 0.9641796946525574
Потери на test на эпохе 1600 = 0.8812092542648315

Потери на train на эпохе 1800 = 0.9389581084251404
Потери на test на эпохе 1800 = 0.8594788908958435

Потери на train на эпохе 2000 = 0.9004624485969543
Потери на test на эпохе 2000 = 0.8384236693

(array([0.6, 0.8, 0.6, 0.8, 0.6, 0.4, 0.8, 0.8, 1. , 0.4, 0.8, 0.4, 0.6,
        0.4, 1. , 0.4, 0.8, 0.8, 0.8, 0.8, 0.2, 1. , 0.8, 1. , 0.2, 0.8,
        0.6, 0.8, 1. , 0.8, 0.6, 0.6, 0.6, 0.4, 0.4], dtype=float32),
 array([0.6916385 , 0.65911657, 0.83671767, 0.5566469 , 0.9412691 ,
        1.197446  , 0.86373234, 0.5266617 , 0.61686295, 0.78639406,
        0.6847644 , 0.6275471 , 0.6648397 , 0.4245677 , 0.76521856,
        0.87554246, 1.0649438 , 0.6735658 , 0.5109702 , 0.5961107 ,
        0.7627705 , 0.65486634, 0.59362674, 1.146484  , 0.35098028,
        0.9649338 , 0.59697205, 0.77334946, 0.6507928 , 0.9885053 ,
        0.98748744, 0.7095967 , 0.49604028, 0.626486  , 0.49815267],
       dtype=float32),
 0.07928977906703949)

Пожалуй, стоит провести более осмысленное сравнение двух датасетов...

Для этого **выполним 10 k-разбиений датасета и сравним средние и разброс**:

In [28]:
kf = KFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)

res1 = []
res2 = []

for train_idx, test_idx in kf.split(dataset_input1):
    kf_input1_train, kf_input1_test = dataset_input1.iloc[train_idx], dataset_input1.iloc[test_idx]
    kf_input2_train, kf_input2_test = dataset_input2.iloc[train_idx], dataset_input2.iloc[test_idx]
    kf_output_train, kf_output_test = dataset_output.iloc[train_idx], dataset_output.iloc[test_idx]

    scaler1 = StandardScaler()
    kf_input1_train = scaler1.fit_transform(kf_input1_train)
    kf_input1_test = scaler1.transform(kf_input1_test)
    scaler2 = StandardScaler()
    kf_input2_train = scaler2.fit_transform(kf_input2_train)
    kf_input2_test = scaler2.transform(kf_input2_test)

    _, _, loss = learning_pipeline(
        Dataset1Model(),
        kf_input1_train,
        kf_output_train,
        kf_input1_test,
        kf_output_test,
        comments=False
    )
    res1.append(loss)

    _, _, loss = learning_pipeline(
        Dataset2Model(),
        kf_input2_train,
        kf_output_train,
        kf_input2_test,
        kf_output_test,
        comments=False
    )
    res2.append(loss)


print(f'Первый датасет -- Avg {np.mean(res1)}, Std {np.std(res1)}')
print(f'Второй датасет -- Avg {np.mean(res2)}, Std {np.std(res2)}')

Лучшая эпоха: 5220 (потери на test: 0.12141019850969315)
Лучшая эпоха: 3211 (потери на test: 0.20064443349838257)
Лучшая эпоха: 2280 (потери на test: 0.14006905257701874)
Лучшая эпоха: 10530 (потери на test: 0.08723389357328415)
Лучшая эпоха: 5498 (потери на test: 0.035534828901290894)
Лучшая эпоха: 4355 (потери на test: 0.028525657951831818)
Лучшая эпоха: 5921 (потери на test: 0.4088318347930908)
Лучшая эпоха: 10387 (потери на test: 0.0533173494040966)
Лучшая эпоха: 9590 (потери на test: 0.048252642154693604)
Лучшая эпоха: 1934 (потери на test: 0.06513302773237228)
Лучшая эпоха: 22447 (потери на test: 0.05099791660904884)
Лучшая эпоха: 2991 (потери на test: 0.08463555574417114)
Лучшая эпоха: 19607 (потери на test: 0.07060530036687851)
Лучшая эпоха: 23334 (потери на test: 0.12964409589767456)
Лучшая эпоха: 20587 (потери на test: 0.05492726340889931)
Лучшая эпоха: 11468 (потери на test: 0.06107795238494873)
Лучшая эпоха: 19136 (потери на test: 0.0519862174987793)
Лучшая эпоха: 18952 (по

Как видно выше, спектральные характеристики заметно точнее восстанаваливают гедонические оценки, нежели средние значения по областям мозга (однако и точность последних достаточно высока).

Но всё ещё остаётся главный вопрос...

### 5. Имеет ли смысл ансамблирование этих джвух моделей???

In [29]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)


res_real = []
res_pred1 = []
res_pred2 = []

for train_idx, test_idx in kf.split(dataset_input1):
    kf_input1_train, kf_input1_test = dataset_input1.iloc[train_idx], dataset_input1.iloc[test_idx]
    kf_input2_train, kf_input2_test = dataset_input2.iloc[train_idx], dataset_input2.iloc[test_idx]
    kf_output_train, kf_output_test = dataset_output.iloc[train_idx], dataset_output.iloc[test_idx]

    scaler1 = StandardScaler()
    kf_input1_train = scaler1.fit_transform(kf_input1_train)
    kf_input1_test = scaler1.transform(kf_input1_test)
    scaler2 = StandardScaler()
    kf_input2_train = scaler2.fit_transform(kf_input2_train)
    kf_input2_test = scaler2.transform(kf_input2_test)

    real, pred1, loss1 = learning_pipeline(
        Dataset1Model(),
        kf_input1_train,
        kf_output_train,
        kf_input1_test,
        kf_output_test,
        comments=False
    )

    _, pred2, loss2 = learning_pipeline(
        Dataset2Model(),
        kf_input2_train,
        kf_output_train,
        kf_input2_test,
        kf_output_test,
        comments=False
    )

    res_pred1.append(pred1)
    res_pred2.append(pred2)
    res_real.append(real)


y = np.concatenate(res_real)
p1 = np.concatenate(res_pred1)
p2 = np.concatenate(res_pred2)

Лучшая эпоха: 7005 (потери на test: 0.21581807732582092)
Лучшая эпоха: 11079 (потери на test: 0.07467319816350937)
Лучшая эпоха: 3015 (потери на test: 0.10074514150619507)
Лучшая эпоха: 6781 (потери на test: 0.07333756238222122)
Лучшая эпоха: 17980 (потери на test: 0.06929344683885574)
Лучшая эпоха: 3808 (потери на test: 0.06358863413333893)
Лучшая эпоха: 20688 (потери на test: 0.07703740894794464)
Лучшая эпоха: 15603 (потери на test: 0.1670844405889511)
Лучшая эпоха: 5992 (потери на test: 0.04222552478313446)
Лучшая эпоха: 4371 (потери на test: 0.04296155273914337)


In [30]:
np.corrcoef(y - p1, y - p2)

array([[1.       , 0.6221718],
       [0.6221718, 1.       ]])

In [31]:
mse_1 = np.mean((y - p1) ** 2)
mse_2 = np.mean((y - p2) ** 2)
mse_1and2 = np.mean((y - (p1 + p2) / 2) ** 2)

print(mse_1, mse_2, mse_1and2)

0.101539694 0.08469196 0.07543546


### **Вывод:** хотя признаки, извлечённые sLORETA, и проигрывают спектральным характеристикам, применение двух моделей в виде ансамбля даёт существенный прирост точности.